# Workspaces

Notebook 08 configured `EpiScopeRuntime` by hand. A **workspace** is the other
way to configure it: a self-contained folder holding an `episcope.toml`, the
papers, the index, the metadata store, saved outputs, and any declarative tasks.
Point EpiScope at the folder and everything else follows from it.

Workspaces are usually met through the CLI (`episcope init`, then run commands
from inside the folder), but they are plain Python objects underneath. This
notebook uses that Python API so you can see what the CLI is doing.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Locate `notebooks/episcope_nb.py`, the shared helper module. This works whether
# the kernel starts in `notebooks/` or at the repository root.
_cwd = Path.cwd()
_nb_dir = next(
    (
        directory
        for candidate in [_cwd, *_cwd.parents]
        for directory in (candidate, candidate / "notebooks")
        if (directory / "episcope_nb.py").is_file()
    ),
    None,
)
if _nb_dir is None:
    raise FileNotFoundError("Could not find notebooks/episcope_nb.py")
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

# Importing the helper also puts `src/` on sys.path when this is a checkout.
import episcope_nb as nb

WORK_DIR = nb.bootstrap("episcope-workspace-")
WORK_DIR

## Create A Workspace

`create_workspace` is exactly what `episcope init <path>` calls. It creates the
directory skeleton and writes `episcope.toml` with local-first defaults — a file
index and an in-memory/JSON metadata store, so nothing external is required.

In [ ]:
from episcope.workspace import create_workspace, find_workspace, load_workspace

workspace = create_workspace(WORK_DIR / "flu-review", name="flu-review")

print("name:", workspace.name)
print("root:", workspace.root)
print("strategy_name:", workspace.strategy_name)
print("\ncontents:")
for entry in sorted(workspace.root.iterdir()):
    print(f"  {entry.name}{'/' if entry.is_dir() else ''}")

`episcope.toml` is the whole configuration. Note what is *not* in it: no API
keys, no `MONGO_URI`, no credentials of any kind. Secrets belong in the
environment (or a `.env`); only local paths and defaults belong here, because the
workspace folder is the thing people copy, share, and commit.

In [ ]:
print(workspace.config_path.read_text())

## Auto-Discovery

This is why the CLI feels implicit: `find_workspace` walks *up* from a starting
directory looking for `episcope.toml`. Run `episcope classify ...` from anywhere
inside the workspace — including a subfolder — and it finds the config. Run it
from outside and it finds nothing, which is when you need `--workspace <path>`.

In [ ]:
nested = workspace.root / "papers" / "2024" / "q1"
nested.mkdir(parents=True, exist_ok=True)

found = find_workspace(nested)
print("from a nested subfolder:", found.name if found else None)

outside = find_workspace(WORK_DIR)
print("from outside the workspace:", outside)

# `load_workspace` is the explicit form -- what `--workspace <path>` uses.
print("loaded explicitly:", load_workspace(workspace.root).name)

## From Workspace To Runtime

`runtime_config_for_workspace` translates a `WorkspaceConfig` into the
`RuntimeConfig` from notebook 08. This is the join between the two: everything
notebook 08 set by hand is now derived from `episcope.toml`.

Watch what happens to the backend fields. The workspace declares
`index_backend = "file"` and `metadata_backend = "memory"`, so `qdrant_url` and
`mongo_uri` come out as `None` and the runtime takes its local-first path —
*even though* `episcope.toml` carries a `qdrant_url` default. That value is only
used if you switch `index_backend` to `"qdrant"`.

In [ ]:
from episcope.services import EpiScopeRuntime
from episcope.services.studio import runtime_config_for_workspace

runtime_config = runtime_config_for_workspace(workspace)

print("strategy_name: ", runtime_config.strategy_name)
print("qdrant_url:    ", runtime_config.qdrant_url)
print("mongo_uri:     ", runtime_config.mongo_uri)
print("retrieval_mode:", runtime_config.retrieval_mode)
print("index_dir:     ", runtime_config.index_dir.relative_to(workspace.root))
print("metadata:      ", runtime_config.metadata_backup.relative_to(workspace.root))
print("workflow_top_k:", runtime_config.workflow_top_k)

## Fill The Workspace

Normally you would drop PDFs into `papers/` and run `episcope index`, which
parses them and writes both the vector index and the metadata store. Here the
helper writes the same two artefacts directly into the paths the workspace
declares, so the notebook stays offline.

In [ ]:
sample_papers = nb.sample_papers()
embedder = nb.TinyKeywordEmbedder(nb.SHARING_VOCABULARY)

vdb, retriever = nb.build_index(runtime_config.index_dir, sample_papers, embedder)
nb.build_academic_db(
    sample_papers,
    runtime_config.strategy_name,
    backup_file=runtime_config.metadata_backup,
)

for entry in sorted(workspace.root.rglob("*")):
    if entry.is_file() and "papers" not in entry.parts:
        print(entry.relative_to(workspace.root))

Now the wiring is observable. `runtime.build_db()` takes no arguments — it reads
`metadata_backup` from the config, which came from `episcope.toml`, and finds the
papers that were just written there.

In [ ]:
runtime = EpiScopeRuntime(runtime_config)

db = runtime.build_db()
print(type(db).__name__, "->", db.list_docs(runtime_config.strategy_name))

The retriever still has to be passed in, for the same reason as in notebook 08:
`build_retriever()` would ask `EmbedderFactory` for the workspace's
`embed_model`, and this notebook's stand-in embedder is not a real model. With a
genuine `embed_model` in `episcope.toml`, `runtime.explore(...)` needs no
overrides at all.

In [ ]:
result = runtime.explore(
    "Is the dataset available in a public repository?",
    top_k=2,
    retriever=retriever,
)

for chunk in result.retrieved_chunks:
    print(f"- {chunk.paper_id} | {chunk.section_title} ({chunk.similarity_score:.3f})")

## Workspace Tasks

`<workspace>/tasks/*.json` is the third delivery path for the declarative specs
from notebooks 06 and 07 — alongside `--task-file` and the API's inline `task`
field. Drop a spec in the folder and it becomes selectable by key, with no
Python and no restart.

`TaskService` is what reads them. Note that it *validates* on read: it builds the
config and renders the prompt, so a malformed spec fails here rather than
halfway through a run.

In [ ]:
import json

from episcope.services.studio import TaskService

funding_spec = {
    "key": "find_funding_sources",
    "kind": "miner",
    "label": "Funding sources",
    "description": "Grants, agencies, and sponsors that funded the study.",
    "top_k": 4,
    "retrieval_templates": [
        "Who funded this study? Which grants or awards supported the work?",
        "Which agencies, foundations, or sponsors are acknowledged?",
    ],
}

tasks_dir = workspace.root / "tasks"
(tasks_dir / "find_funding_sources.json").write_text(json.dumps(funding_spec, indent=2))

tasks = TaskService(workspace)
print("specs on disk:", [spec.key for spec in tasks.specs()])

print("\nminers visible to the CLI, API, and UI:")
for entry in tasks.list()["miners"]:
    print(f"  {entry['key']:28s} {entry['source']}")

In [ ]:
# Validation failures are reported in plain language rather than as a raw
# pydantic traceback. `episcope tasks validate --task-file ...` prints the same.
try:
    TaskService.validate({"key": "Bad Key", "kind": "miner"})
except ValueError as exc:
    print(exc)

## The CLI Equivalents

Everything above has a command-line form, which is how workspaces are normally
used:

| Python | CLI |
| --- | --- |
| `create_workspace(path, name=...)` | `episcope init <path> --name <name>` |
| `find_workspace()` | implicit — run any command from inside the workspace |
| `load_workspace(path)` | `--workspace <path>` on any command |
| writing `tasks/*.json` | `episcope tasks new --kind miner > tasks/funding.json` |
| `TaskService(ws).list()` | `episcope tasks --workspace <path>` |
| `TaskService.validate(spec)` | `episcope tasks validate --task-file <file>` |
| indexing into `index_dir` | `episcope index` (parses `papers/`) |
| `runtime.explore(...)` | `episcope explore "..."` / `episcope ask "..."` |

`episcope studio` opens the Streamlit UI over a workspace, and
`WorkspaceService` (in `episcope.services.studio`) is the multi-workspace manager
it uses to create, list, and update them under a common root.

## Summary

A workspace is configuration-as-a-folder. It gives you a portable unit of work
with paths that resolve relative to its own root, local-first storage defaults, a
drop-in location for declarative tasks, and auto-discovery so commands do not
need repeating arguments.

Two rules worth keeping: **secrets never go in `episcope.toml`** — only paths and
defaults — and the backend fields (`index_backend`, `metadata_backend`) decide
whether the runtime goes local-first or reaches for Qdrant and MongoDB, not the
presence of a URL in the file.